Assess how well top-ranked ligands can predict a gene set of interest
================

This tutorial assesses the ligands prioritized by NicheNet in their
ability to predict a gene set of interest. We will first follow the
steps of [Perform NicheNet analysis starting from an AnnData
object](wrapper.ipynb)) to obtain ligands rankings. Make sure you
understand the steps and output of a basic NicheNet analysis (more
information in [Perform NicheNet analysis starting from an AnnData object:
step-by-step analysis](steps.ipynb). You can also apply this
tutorial to the [NicheNet’s ligand activity analysis on a gene set of
interest](ligand_activity_geneset.ipynb) notebook.


In [1]:
from nichenetpy.wrappers import (
    run_nichenet,
    calculate_fraction_top_predicted,
    calculate_fraction_top_predicted_fisher,
    get_top_predicted_genes
)
from nichenetpy.gene_symbol import mouse_alias_info
from nichenetpy.prediction import assess_rf_class_probabilities
from nichenetpy.metrics import calculate_metrics

import anndata
import os
import requests
import pickle
import pandas as pd
import session_info

Download the model pickle

In [2]:
filename = "nichenet_mouse.pkl"
file_path = os.path.join("./tutorial_files", filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14887637/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Download the AnnData object

In [3]:
data_path = os.path.normpath("./tutorial_files/AnnData")
if not os.path.exists(data_path):
    os.makedirs(data_path)
filename = "annData3531889.h5"
file_path = os.path.join(data_path, filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14859451/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Perform nichenet analysis using the wrapper function

In [4]:
ann = anndata.io.read_h5ad(os.path.join(data_path, "annData3531889.h5"))
ann.var_names = ann.var["gene"]
mouse_alias_info.alias_to_symbol(ann)
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor = model["predictor"]
lr_network = model["lr_network"]
lr_sig = model["lr_sig"]
receiver = "CD8 T"
sender_celltypes = ["CD4 T","Treg", "Mono", "NK", "B", "DC"]
output = run_nichenet(
    ann,
    predictor,
    lr_network,
    lr_sig=lr_sig,
    receiver=receiver,
    sender_celltypes=sender_celltypes,
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    expression_pct=0.05,
    targets_top_n=100
)
geneset_oi = output["geneset_oi"]
expressed_genes_receiver = output["expressed_genes_receiver"]
ligands_oi = output["best_upstream_ligands"]

## Assess how well top-ranked ligands can predict a gene set of interest

For the top 30 ligands, we will now build a multi-ligand model that uses
all top-ranked ligands to predict whether a gene belongs to the gene set
of interest (differentially expressed genes in CD8 T cells after LCMV
infection) or not. This classification model will be trained via
cross-validation and returns a probability for every gene.

In [5]:
n = 2
k = 3
gene_predictions_top30_list = [
    assess_rf_class_probabilities(
        folds=k,
        geneset=geneset_oi,
        background_expressed_genes=expressed_genes_receiver,
        ligands_oi=ligands_oi,
        predictor=predictor
    ) for _ in range(n)
]

Evaluate how well the target gene probabilities accord to the gene set assignments.

In [6]:
target_prediction_performances = []
for df in gene_predictions_top30_list:
    met = calculate_metrics(list(df["prediction"]), list(df["response"]))
    target_prediction_performances.append(pd.DataFrame([list(met.values())], columns=met.keys()))
target_prediction_performances = pd.concat(target_prediction_performances)
target_prediction_performances.reset_index(drop=True, inplace=True)

What is the AUROC, AUPR and PCC of this model (averaged over cross-validation rounds)?

In [7]:
target_prediction_performances.mean()

auroc             0.821206
pearson           0.552095
aupr              0.491261
aupr_corrected    0.415736
dtype: float64

Evaluate whether genes belonging to the gene set are more likely to be top-predicted. We will look at the top 5% of predicted targets here.

In [8]:
target_prediction_performances_discrete = pd.concat([calculate_fraction_top_predicted(df) for df in gene_predictions_top30_list])
target_prediction_performances_discrete

,true_target,n,positive_prediction,fraction_positive_predicted
0,0,2950,49,0.016610
1,1,241,112,0.464730
0,0,2950,49,0.016610
1,1,241,111,0.460581


What is the fraction of viral response genes that belongs to the top 5% predicted targets?

In [9]:
target_prediction_performances_discrete[
    target_prediction_performances_discrete["true_target"] == 1
]["fraction_positive_predicted"].mean()

np.float64(0.46265560165975106)

What is the fraction of non-viral-response genes that belongs to the top 5% predicted targets?

In [10]:
target_prediction_performances_discrete[
    target_prediction_performances_discrete["true_target"] == 0
]["fraction_positive_predicted"].mean()

np.float64(0.016610169491525422)

We see that the viral response genes are enriched in the top-predicted target genes. To test this, we will now apply a Fisher’s exact test for every cross-validation round and report the average p-value.

In [11]:
target_prediction_performances_discrete_fisher = [
    calculate_fraction_top_predicted_fisher(e).pvalue for e in gene_predictions_top30_list
]
sum(target_prediction_performances_discrete_fisher)/len(target_prediction_performances_discrete_fisher)

np.float64(3.4436529531004207e-97)

Finally, we will look at which p-EMT genes are well-predicted in every cross-validation round.

In [12]:
dfs = []
for round, affected_gene_predictions in enumerate(gene_predictions_top30_list):
    df = get_top_predicted_genes(affected_gene_predictions)
    df.rename(columns={"predicted_top_target": f"predicted_top_target_round{round}"}, inplace=True)
    dfs.append(df)
top_predicted_genes = dfs[0]
for df in dfs[1:]:
    top_predicted_genes = top_predicted_genes.merge(df, on=["gene", "true_target"], how="outer")
top_predicted_genes[top_predicted_genes["true_target"] == 1]

,gene,true_target,predicted_top_target_round0,predicted_top_target_round1
5,1600014C10Rik,1,False,False
14,2410006H16Rik,1,False,False
43,Acadl,1,False,False
56,Actb,1,False,False
68,Adar,1,True,True
...,...,...,...,...
3062,Vim,1,False,False
3103,Xaf1,1,True,True
3106,Xpo1,1,False,False
3129,Zbp1,1,True,True


### Comparison between seurat and scanpy

We now compare the reproduction of seurat's FindAllMarkers to the output when using scanpy's rank_genes_groups. 

In [13]:
output = run_nichenet(
    ann,
    predictor,
    lr_network,
    lr_sig=lr_sig,
    receiver=receiver,
    sender_celltypes=sender_celltypes,
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    expression_pct=0.05,
    targets_top_n=100,
    use_scanpy=True
)
geneset_oi = output["geneset_oi"]
expressed_genes_receiver = output["expressed_genes_receiver"]
ligands_oi = output["best_upstream_ligands"]

In [14]:
n = 2
k = 3
gene_predictions_top30_list_scanpy = [
    assess_rf_class_probabilities(
        folds=k,
        geneset=geneset_oi,
        background_expressed_genes=expressed_genes_receiver,
        ligands_oi=ligands_oi,
        predictor=predictor
    ) for _ in range(n)
]

In [15]:
target_prediction_performances_scanpy = []
for df in gene_predictions_top30_list_scanpy:
    met = calculate_metrics(list(df["prediction"]), list(df["response"]))
    target_prediction_performances_scanpy.append(pd.DataFrame([list(met.values())], columns=met.keys()))
target_prediction_performances_scanpy = pd.concat(target_prediction_performances_scanpy)
target_prediction_performances_scanpy.reset_index(drop=True, inplace=True)

use_scanpy=True leads to slightly lower performance according to the metrics below

In [16]:
target_prediction_performances.mean()

auroc             0.821206
pearson           0.552095
aupr              0.491261
aupr_corrected    0.415736
dtype: float64

In [17]:
target_prediction_performances_scanpy.mean()

auroc             0.774296
pearson           0.496145
aupr              0.432964
aupr_corrected    0.360682
dtype: float64

with use_scanpy=True, we get both a higher false-positive rate and a lower true-positive rate

In [18]:
target_prediction_performances_discrete

,true_target,n,positive_prediction,fraction_positive_predicted
0,0,2950,49,0.016610
1,1,241,112,0.464730
0,0,2950,49,0.016610
1,1,241,111,0.460581


In [19]:
target_prediction_performances_discrete_scanpy = pd.concat([calculate_fraction_top_predicted(df) for df in gene_predictions_top30_list_scanpy])
target_prediction_performances_discrete_scanpy

,true_target,n,positive_prediction,fraction_positive_predicted
0,0,2952,63,0.021341
1,1,230,97,0.421739
0,0,2952,65,0.022019
1,1,230,95,0.413043


In [20]:
session_info.show()